In [ ]:
!pip install streamlit pyngrok pandas matplotlib reportlab -q

import os
os.makedirs("fair-lens", exist_ok=True)

!wget -O fair-lens/compas.csv https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv

--2026-05-02 09:18:09--  https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2546489 (2.4M) [text/plain]
Saving to: ‘fair-lens/compas.csv’

fair-lens/compas.cs 100%[===================>]   2.43M  --.-KB/s    in 0.04s   

2026-05-02 09:18:09 (59.5 MB/s) - ‘fair-lens/compas.csv’ saved [2546489/2546489]



In [1]:
!pip install streamlit pyngrok pandas matplotlib reportlab -q

import os
os.makedirs("fair-lens", exist_ok=True)

!wget -O fair-lens/compas.csv https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 103.0 MB/s eta 0:00:00
--2026-05-02 09:20:44--  https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2546489 (2.4M) [text/plain]
Saving to: ‘fair-lens/compas.csv’

fair-lens/compas.cs 100%[===================>]   2.43M  --.-KB/s    in 0.04s   

2026-05-02 09:20:45 (69.0 MB/s) - ‘fair-lens/compas.csv’ saved [2546489/2546489]



In [2]:
%%writefile fair-lens/app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
import io

st.set_page_config(page_title="Fair-Lens AI Bias Dashboard", layout="wide")

st.title("⚖️ Fair-Lens: AI Fairness & Bias Dashboard")
st.markdown("### COMPAS Dataset Analysis with Explainable Insights")

# Load Data
@st.cache_data
def load_data():
    return pd.read_csv("fair-lens/compas.csv")

df = load_data()

# Sidebar
st.sidebar.header("Filters")

if "race" in df.columns:
    race = st.sidebar.selectbox("Select Race", df["race"].unique())
    df = df[df["race"] == race]

st.sidebar.write("Rows:", len(df))

# Charts
col1, col2 = st.columns(2)

with col1:
    st.subheader("📊 Risk Score Distribution")
    if "decile_score" in df.columns:
        fig, ax = plt.subplots()
        df["decile_score"].hist(bins=10, ax=ax)
        st.pyplot(fig)

with col2:
    st.subheader("👥 Group Distribution")
    if "race" in df.columns:
        fig, ax = plt.subplots()
        df["race"].value_counts().plot(kind="bar", ax=ax)
        st.pyplot(fig)

# Fairness Metric
st.markdown("---")
st.subheader("⚖️ Fairness Insight")

if "two_year_recid" in df.columns:
    rate = df["two_year_recid"].mean()
    st.metric("Recidivism Rate", f"{rate:.2f}")
else:
    rate = 0

# AI Explanation Box
st.markdown("---")
st.subheader("🤖 AI Explanation")

if rate > 0.6:
    explanation = """
    This subgroup shows a high recidivism prediction rate.
    This may indicate possible bias amplification or historical imbalance.
    Further fairness auditing is strongly recommended.
    """
elif rate > 0.3:
    explanation = """
    This subgroup shows a moderate recidivism prediction rate.
    Results should be interpreted carefully and compared across demographic groups.
    """
else:
    explanation = """
    This subgroup shows a relatively low recidivism prediction rate.
    However, fairness validation remains essential before deployment.
    """

st.info(explanation)

# PDF Generator
def generate_pdf(rate, rows):
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(buffer)
    styles = getSampleStyleSheet()
    content = []

    content.append(Paragraph("Fair-Lens Report", styles["Title"]))
    content.append(Spacer(1, 12))
    content.append(Paragraph(f"Recidivism Rate: {rate:.2f}", styles["Normal"]))
    content.append(Spacer(1, 12))
    content.append(Paragraph(f"Total Records: {rows}", styles["Normal"]))
    content.append(Spacer(1, 12))
    content.append(Paragraph(explanation, styles["Normal"]))

    doc.build(content)
    buffer.seek(0)
    return buffer

pdf = generate_pdf(rate, len(df))

st.download_button(
    "📥 Download PDF Report",
    data=pdf,
    file_name="fair_lens_report.pdf",
    mime="application/pdf"
)

# Data Preview
st.markdown("---")
st.subheader("📄 Data Preview")
st.dataframe(df.head(20))

Writing fair-lens/app.py


In [3]:
from pyngrok import ngrok
import os, time

ngrok.set_auth_token("3D7sGIL3fljbvySai1QgfcBjjzf_5QmeqqQyDQGssyM8eSoZE")

os.system("streamlit run fair-lens/app.py --server.address 0.0.0.0 --server.port 8501 &")

time.sleep(10)

public_url = ngrok.connect(8501, "http")
print("🔥 LIVE LINK:")
print(public_url)

🔥 LIVE LINK:
NgrokTunnel: "https://monsieur-unfrozen-amusement.ngrok-free.dev" -> "http://localhost:8501"
